# CHSH quantum empirical model

The goal is to show how the CHSH quantum realization can be implemented and verified within the contextuality package.

As a reminder, the CHSH measurement scenario consists of two observers: Alice and Bob. Each of these observers, have two possible measurements, A0 and A1 for Alice and B0 and B1 for Bob.

A system is prepared with two qubits in a certain state $|\psi \rangle$, and then one qubit is sent to Alice and one to Bob, who perform an observable of the type:

$$ \langle A_x \otimes B_y \rangle $$

Where $\otimes$ represents the tensor product.

## Imports

In [19]:
from qutip import basis, sigmax, sigmaz, tensor, ket2dm, identity
import numpy as np
from contextuality.measurement_scenario import MeasurementScenarioImplementations
from contextuality.empirical_model import EmpiricalModel

## Defining the quantum objects

From the documentation of the method `quantum_realisation`, in `empirical_model.py` we have:
```python
    def quantum_realisation(self, rho: ArrayLike, pvms: ArrayLike) -> None:
        r"""
        Compute an empirical model/behavior from a provided quantum realization.

        :param rho:     The quantum state density matrix.
        :param pvms:    The measurements in an array. The indices are "measurement label", "outcome" to access
                        a specific measurement PVM. For instance, meas[0,0] accesses the PVM for measurement
                        with label X[0] and outcome O[0] respectively.
        """
    ...
```

Which means that our PVMs must be in the form:
- $A_x^i \otimes I_2$ for Alice's measurements and
- $I_2 \otimes B_y^i$ for Bob's measurements.

Where the $i$ refers to the outcome ($i \in \{-1,1\}$).

We take the quantum realisation which gives the Tsirelson bound, i.e., the maximum quantum bound.

In [20]:
zero, one = basis(2,0), basis(2,1)

psi = (tensor(zero, one) - tensor(one, zero)) / np.sqrt(2)
rho = ket2dm(psi).unit().full()

sz, sx = sigmaz(), sigmax()

A0 = [ket2dm(x) for x in sz.eigenstates()[1]]
A1 = [ket2dm(x) for x in sx.eigenstates()[1]]
B0 = [ket2dm(x) for x in (-(sx + sz) / np.sqrt(2)).eigenstates()[1]]
B1 = [ket2dm(x) for x in ((sx - sz) / np.sqrt(2)).eigenstates()[1]]

PVMs = [
    [tensor(a, identity(2)).full() for a in A0],
    [tensor(a, identity(2)).full() for a in A1],
    [tensor(identity(2), b).full() for b in B0],
    [tensor(identity(2), b).full() for b in B1]
]

## Creating the empirical model and the measurement scenario

In [21]:
chsh_ms = MeasurementScenarioImplementations.CHSH()
quantum_em = EmpiricalModel(chsh_ms)
quantum_em.quantum_realisation(rho, PVMs)

print(quantum_em)

EmpiricalModel(MeasurementScenario(X=[0, 1, 2, 3], M=[[0, 2], [0, 3], [1, 2], [1, 3]], O=[0, 1])
	0.43 0.07 0.07 0.43 
	0.43 0.07 0.07 0.43 
	0.43 0.07 0.07 0.43 
	0.07 0.43 0.43 0.07 
)


## Measuring the contextual fraction

In [22]:
cf = quantum_em.compute_cf()['CF']

print(cf) # The known CF is sqrt(2) - 1 ~= 0.41

0.41421356237309515
